#  Text Classification with RoBERTa

## Objective

This notebook aims to:

- Train a text classification model (**roberta-base**) on our dataset  
-  Predict whether a review is **positive**, **negative**, or **neutral** using our trained model
- Evaluate the model by calculating its **accuracy**  
- Save the trained model for **future use**

##  Training Process

The training process consists of the following steps:

1. Import the balanced dataset and extract the input data along with their corresponding labels  
2. Split the dataset into training and testing sets  
3. Import the `Trainer` from the Transformers library (responsible for handling the training process)  
4. Prepare the training and testing datasets to be passed as arguments to the `Trainer`

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [18]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import  accuracy_score
import torch
import pandas as pd
import os

In [3]:
os.getcwd()

'/content'

In [4]:
os.chdir("/content/drive/MyDrive/transformers/Ecommerce/CustomerFeedbackAnalysis")

In [5]:
os.getcwd()

'/content/drive/MyDrive/transformers/Ecommerce/CustomerFeedbackAnalysis'

In [6]:
! ls

01_data_collection_and_balancing.ipynb.ipynb  ProductsReviews.csv
balanced_dataset.csv			      ProductsReviews.zip
images					      training_model.ipynb


### 1. Load the balanced dataset and import the RoBERTa tokenizer and model

In [7]:
df = pd.read_csv("./balanced_dataset.csv")

In [8]:
df.head()

,reviews.text,sentiment,class
0,Delivered on time and it looked good will hook...,Positive,2
1,The price for this tablet is not the only grea...,Positive,2
2,Perfect item for any place in your house. Love...,Positive,2
3,My husband loves this Kindle. He has a muscula...,Positive,2
4,The Amazon Kindle is a series of e-readers des...,Positive,2


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293 entries, 0 to 292
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   reviews.text  293 non-null    object
 1   sentiment     293 non-null    object
 2   class         293 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 7.0+ KB


In [21]:
model_name = "roberta-base"

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained(model_name)

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels= 3)

In [26]:
prompt= "it is a bad product"

In [27]:
tokenized_prompt = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True , max_length= 512)

In [28]:
output = model(**tokenized_prompt)

In [29]:
logits = output.logits

In [30]:
logits

tensor([[ 0.0982, -0.0405,  0.0472]], grad_fn=<AddmmBackward0>)

In [31]:
probs = torch.softmax(logits, dim= 1)

In [32]:
probs # we see the same probabilities , this why we need to train the model

tensor([[0.3545, 0.3086, 0.3369]], grad_fn=<SoftmaxBackward0>)

### 2. Prepare the training and testing datasets for the Trainer

A training dataset passed to the `Trainer` is structured as a list of dictionaries.

Each dictionary represents one input example and should contain:

- `input_ids`: tokenized representation of the input text  
- `attention_mask`: indicates which tokens should be attended to  
- `labels`: the label corresponding to the input  

### Example

```python
[
  {
    "input_ids": [...],
    "attention_mask": [...],
    "labels": 2
  },
  {
    "input_ids": [...],
    "attention_mask": [...],
    "labels": 0
  }
]

In [10]:
input_texts = df["reviews.text"].tolist()

In [13]:
labels = df["class"].tolist()

In [14]:
len(input_texts), len(labels)

(293, 293)

In [15]:
train_inputs, test_inputs, train_labels, test_labels  = train_test_split(input_texts, labels, test_size=0.2, stratify=labels) # split data : 20% test, 80% train

In [16]:
len(train_inputs), len(test_inputs)

(234, 59)

In [17]:
type(train_labels)

list

In [19]:
train_labels = torch.tensor(train_labels)

In [20]:
test_labels = torch.tensor(test_labels)

In [33]:
tokenized_train_inputs = tokenizer(train_inputs, return_tensors="pt", padding=True, truncation=True , max_length= 512)
tokenized_test_inputs = tokenizer(test_inputs, return_tensors="pt", padding=True, truncation=True , max_length= 512)

In [34]:
tokenized_test_inputs # A dictionary-like object with two keys: "input_ids" and "attention_mask"

{'input_ids': tensor([[    0,   387, 12807,  ...,     1,     1,     1],
        [    0,   133,  6173,  ...,  7058,     4,     2],
        [    0, 40084,   209,  ...,     1,     1,     1],
        ...,
        [    0,   428, 12807,  ...,     1,     1,     1],
        [    0,   100,    38,  ...,     1,     1,     1],
        [    0, 12350,  9995,  ...,     1,     1,     1]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [35]:
train_dataset = []
for i in range(len(train_inputs)):
  input_ids = tokenized_train_inputs["input_ids"][i]
  attention_mask = tokenized_train_inputs["attention_mask"][i]
  label = train_labels[i].item()
  train_dataset.append({"input_ids": input_ids, "attention_mask": attention_mask, "labels": label})

In [36]:
test_dataset = []
for i in range(len(test_inputs)):
  input_ids = tokenized_test_inputs["input_ids"][i]
  attention_mask = tokenized_test_inputs["attention_mask"][i]
  label = test_labels[i].item()
  test_dataset.append({"input_ids": input_ids, "attention_mask": attention_mask, "labels": label})

### 3. Prepare the training arguments and the Trainer… then sit back and wait for the training process (this might take a while ☕)

In [37]:
training_args = TrainingArguments(
    output_dir="./results" , # save logs and configuration file related to the training process
    per_device_train_batch_size= 5 , # batch size(number of samples per step) => chunking process
    num_train_epochs= 8 ,
    eval_strategy= "epoch" ,
    report_to = "none"
)

In [38]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [39]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    processing_class = tokenizer,
    compute_metrics = compute_metrics
)

In [40]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.986156,0.423729
2,No log,0.697831,0.627119
3,No log,0.894271,0.728814
4,No log,1.934195,0.627119
5,No log,1.908095,0.711864
6,No log,1.843022,0.728814
7,No log,1.888754,0.728814
8,No log,1.928824,0.711864


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=376, training_loss=0.34748442629550363, metrics={'train_runtime': 113.5091, 'train_samples_per_second': 16.492, 'train_steps_per_second': 3.313, 'total_flos': 210679846951392.0, 'train_loss': 0.34748442629550363, 'epoch': 8.0})

## 4. Save the trained model for future use

In [ ]:
trainer.save_model("./FinalModel")

In [42]:
tokenizer.save_pretrained("./FinalModel")

('./FinalModel/tokenizer_config.json', './FinalModel/tokenizer.json')

In [43]:
train_results = trainer.evaluate(train_dataset)
train_accuracy = train_results["eval_accuracy"]

In [44]:
test_results = trainer.evaluate(test_dataset)

In [46]:
print(f"Train Accuracy : {train_results["eval_accuracy"]}")
print(f"Test Accuracy : {test_results["eval_accuracy"]}")

Train Accuracy : 1.0
Test Accuracy : 0.711864406779661


In [47]:
print("hello")

hello
